In [2]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sentence_transformers import SentenceTransformer
import os

c:\Users\Anshul\Desktop\Major Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ---------- CONFIG ----------
INPUT_FILE = "cleaned_pairs.csv"
OUTPUT_DIR = "graph_data"
EMBED_MODEL = "all-MiniLM-L6-v2"  # or "all-mpnet-base-v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# 1. Load cleaned data
df = pd.read_csv(INPUT_FILE)
print("Loaded cleaned dataset:", len(df), "pairs")


Loaded cleaned dataset: 86376 pairs


In [9]:

# 2. Combine text for embeddings
df["email_text"] = df["email_subject_clean"].fillna('') + " " + df["email_body_clean"].fillna('')
df["reply_text"] = df["reply_subject_clean"].fillna('') + " " + df["reply_body_clean"].fillna('')

node_texts = []
node_types = []  # 0 = email, 1 = reply
pair_map = []    # for debugging

for i, row in df.iterrows():
    node_texts.append(row["email_text"])
    node_types.append(0)
    pair_map.append(row["pair_id"])
    node_texts.append(row["reply_text"])
    node_types.append(1)
    pair_map.append(row["pair_id"])


In [11]:
# 3. Generate embeddings
print("Encoding text embeddings...")
model = SentenceTransformer(EMBED_MODEL)
embeddings = model.encode(node_texts, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)


Encoding text embeddings...


Batches: 100%|██████████| 5399/5399 [34:42<00:00,  2.59it/s]  


Embeddings shape: (172752, 384)


In [12]:

# 4. Normalize vectors (recommended)
embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)


In [13]:

# 5. Create edges (email → reply)
edge_index_list = [[], []]
for i in range(0, len(node_texts), 2):
    src = i        # email node
    dst = i + 1    # reply node
    edge_index_list[0].append(src)
    edge_index_list[1].append(dst)

edge_index = torch.tensor(edge_index_list, dtype=torch.long)
x = torch.tensor(embeddings, dtype=torch.float32)


In [14]:

# 6. Optionally, add node_type feature
node_type_tensor = torch.tensor(node_types, dtype=torch.long).unsqueeze(1)
x = torch.cat([x, node_type_tensor.float()], dim=1)  # Append node type as last dim


In [15]:
# 7. Create PyTorch Geometric Data object
data = Data(x=x, edge_index=edge_index)
print(data)
print("Graph created with", data.num_nodes, "nodes and", data.num_edges, "edges.")


Data(x=[172752, 385], edge_index=[2, 86376])
Graph created with 172752 nodes and 86376 edges.


In [16]:

# 8. Save graph artifacts
np.save(os.path.join(OUTPUT_DIR, "embeddings.npy"), embeddings)
np.save(os.path.join(OUTPUT_DIR, "edge_index.npy"), edge_index.numpy())
df.to_csv(os.path.join(OUTPUT_DIR, "graph_pairs.csv"), index=False)
print("✅ Saved graph data to", OUTPUT_DIR)

✅ Saved graph data to graph_data
